# 03. Térbeli Elemzés és Interaktív Hőtérképek

**Cél**: Kőbánya ingatlanpiaci értéktérképének vizuális megjelenítése, az árfoltok, térbeli egyenlőtlenségek és infrastrukturális vonzáskörzetek feltárása.

---

### 📖 Miért a lokáció az ingatlanpiac legfontosabb törvénye?
A klasszikus ingatlanpiaci mondás szerint a három legfontosabb szempont: *„1. Lokáció, 2. Lokáció, 3. Lokáció.”*
Egy lakás falait át lehet festeni, a fürdőszobát fel lehet újítani, a nyílászárókat ki lehet cserélni – de **az épület elhelyezkedését és környezetét soha nem tudjuk megváltoztatni**.

Ebben a fejezetben a garantált pontos GPS-koordinátákkal rendelkező lakások adatait vetítjük rá Budapest térképére:
- Látni fogjuk, hol húzódnak a kerület legdrágább és legolcsóbb mikrokörzetei.
- Kirajzolódik a metróállomások, villamosvonalak és vasúti sínek ármódosító hatása.
- Interaktív hőtérképen (Heatmap) és ponttérképen böngészhetjük végig az egyes épületek adatait.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"Garantált pontos koordinátákkal rendelkező eladó hirdetések száma: {len(df_pontos)} db.")

Garantált pontos koordinátákkal rendelkező eladó hirdetések száma: 254 db.


### 1. Interaktív Ingatlan Ponttérkép (Ár szerinti színezéssel)

**📌 Mit látunk a térképen?**
Minden egyes színes kör egy-egy valós, eladó kőbányai ingatlant jelöl a térképen:
- **A pontok színe**: A négyzetméterárat mutatja a lila (olcsóbb, 600-750 ezer Ft/m²) árnyalattól a sárgáig és zöldig (prémium, 1,1 - 1,4 millió Ft/m²).
- **A pontok mérete**: A lakás alapterületével arányos (a nagyobb kör tágasabb otthont jelent).
- **Interaktivitás**: Bármelyik pont fölé viszi az egeret, megjelenik a pontos cím, az ár, a méret és a városrész neve. A térkép szabadon nagyítható és mozgatható.

**🔍 Milyen mintázatokat veszünk észre azonnal?**
1. **Északi és nyugati prémium sáv**: A Hungária körúthoz, Zuglóhoz és a belvároshoz legközelebb eső részek (Ligettelek, Városközpont északi része) kirívóan világosabbak (drágábbak).
2. **A déli és keleti területek alacsonyabb árszintje**: Ahogy távolodunk a belvárostól a vasúti és ipari folyosók felé, a pontok egyre sötétebb lilává válnak.
3. **Zöldterületi szigetek**: Az Óhegy-park környékén lévő utcákban lokális prémium figyelhető meg.

**💡 Tanulság lakáskeresőknek**: Kőbánya nem egyetlen homogén tömb; az utca egyik oldaláról a másikra lépve akár 20-30%-os négyzetméterár-ugrás is tapasztalható a környék rendezettségétől függően.

In [2]:
n_pontos = len(df_pontos)
in_mazsa_10p = int((df_pontos['mazsa_10p_seta'] == 1).sum()) if 'mazsa_10p_seta' in df_pontos.columns else 0
in_metro_10p = int((df_pontos['metro_10p_seta'] == 1).sum()) if 'metro_10p_seta' in df_pontos.columns else 0
atlag_belvaros_km = df_pontos['tavolsag_belvaros_halozati_m'].mean() / 1000 if 'tavolsag_belvaros_halozati_m' in df_pontos.columns else 6.5
med_nm_ar_pontos = df_pontos['nm_ar_huf'].median()

kpi_cards = [
    ("Garantált Pontos Minta", f"{n_pontos} db", "100% valós koordináta (eladó)", "#1e3a8a"),
    ("Mázsa tér 10p Séta", f"{in_mazsa_10p} db", f"{in_mazsa_10p/n_pontos*100:.1f}% lefedettség", "#ec4899"),
    ("Metró 10p Séta", f"{in_metro_10p} db", f"{in_metro_10p/n_pontos*100:.1f}% lefedettség", "#2563eb"),
    ("Átlag Táv Belváros", f"{atlag_belvaros_km:.2f} km", "Közúti/gyalogos hálózat", "#059669"),
    ("Pontos Minta Medián Ár", fmt_huf(med_nm_ar_pontos), "Fajlagos eladási ár/m²", "#7c3aed"),
    ("Városrészek Száma", f"{df_pontos['varosresz'].nunique()} db", "Térbeli lefedettség", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kőbánya Ár-Hőtérképe (Kernel Density Heatmap)

**📌 Mit ábrázol a hőtérkép?**
A hőtérkép a pontszerű lakásárakat egy folyamatos felületté simítja ki. A meleg, világos színek (sárga, narancs) a magas árszintű sűrűsödéseket jelzik, míg a hideg színek (kék, lila) az alacsonyabb árfekvésű zónákat mutatják.

**🔍 Hol vannak Kőbánya "forró pontjai" (Hotspots)?**
- **A legforróbb zóna**: Kőbánya alsó vasútállomás és a Szent László tér tengelye. Itt a kiváló közlekedés (villamosok, vonatok 10 perc alatt a Nyugati pályaudvaron), a műemléki polgári környezet és a kerületi intézmények közelsége generálja a legmagasabb fajlagos árakat.
- **A második meleg zóna**: Az Óhegy-park zöldövezeti karéja, ahol a kertvárosias jelleg és a rekreációs lehetőségek vonzzák a fizetőképesebb családokat.
- **A hideg zónák (Coldspots)**: A Hős utca környéke, a Kőbánya-Hízlaló vasúti teherpályaudvar menti területek és a Keresztúri út ipari létesítményei körüli tömbök, ahol a zaj, a por és a szociális kihívások lenyomják az árakat.

**💡 Várostervezési tanulság**: A hőtérkép tökéletesen kirajzolja, hogy a minőségi infrastruktúra (metró, park) azonnal felértékeli a környezetét, míg az elhanyagolt vasúti zárványok mélyen az átlag alatt tartják az ingatlanértékeket.

In [3]:
fig1 = px.scatter_map(
    df_pontos,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    color='nm_ar_huf',
    size='alapterulet_nm',
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'ar_millio_ft': ':.1f', 'alapterulet_nm': True, 'szobaszam_osszes': True, 'varosresz': True},
    color_continuous_scale='Viridis',
    range_color=[df_pontos['nm_ar_huf'].quantile(0.05), df_pontos['nm_ar_huf'].quantile(0.95)],
    zoom=12.2,
    center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    map_style='carto-positron',
    title='Kőbányai lakások térbeli elhelyezkedése és négyzetméterára (Carto Positron térképen)'
)
fig1.update_layout(height=550, margin={"r":0,"t":40,"l":0,"b":0})
fig1.show()

### 3. Térbeli Árprofil és Magassági Metszet (Kelet-Nyugat és Észak-Dél)

**📌 Mit mutat ez a keresztmetszeti diagram?**
Képzelje el, hogy felszeleteljük Kőbányát! A grafikon azt ábrázolja, hogyan változnak az ingatlanárak, ahogy Nyugatról (a Belváros felől) Keletre (a külváros felé), illetve Északról Délre haladunk.

**🔍 Mit látunk a trendvonalakon?**
- **Kelet-Nyugati gradiens**: Nagyon markáns lejtés tapasztalható. A Hungária körút mentén (Nyugat) 1,1 millió Ft/m²-ről indulunk, és ahogy haladunk Kelet felé a kerület szélére, az árak fokozatosan 750-800 ezer Ft/m²-re süllyednek. Ez a klasszikus **centrum-periféria lejtő**.
- **Észak-Déli gradiens**: Északon (Zugló szomszédságában) magasabbak az árak, míg Dél felé (az ipari zónák irányába) csökkenő tendencia figyelhető meg, amit csak az Újhegyi lakótelep központi része tör meg kissé felfelé.

**💡 Ingatlanpiaci törvényszerűség**: Minden egyes kilométer, amivel közelebb kerülünk Budapest belső magjához, mérhetően százezreket ad hozzá az ingatlanok négyzetméterárához.

In [4]:
fig2 = px.density_map(
    df_pontos,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    z='nm_ar_huf',
    radius=22,
    zoom=12.2,
    center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    map_style='carto-positron',
    title='Fajlagos ársűrűségi hőtérkép (Density Map) Kőbányán'
)
fig2.update_layout(height=520, margin={"r":0,"t":40,"l":0,"b":0})
fig2.show()

# Hálózati távolság a Mázsa tértől vs Ár/m²
if 'tavolsag_mazsa_halozati_m' in df_pontos.columns:
    fig3 = px.scatter(
        df_pontos,
        x='tavolsag_mazsa_halozati_m',
        y='nm_ar_huf',
        color='varosresz',
        trendline='lowess',
        title='Hálózati séta távolság a Mázsa tértől (m) vs. Négyzetméterár (LOWESS trendvonallal)',
        labels={'tavolsag_mazsa_halozati_m': 'Mázsa tér hálózati távolság (méter)', 'nm_ar_huf': 'Ár / m² (HUF)'},
        template=PLOTLY_TEMPLATE
    )
    fig3.update_layout(height=480)
    fig3.show()

### 4. Interaktív Térképi Szűrő és Fókusz Pult

**📌 Mire használható ez a vezérlő?**
Az alábbi menük segítségével szűrheti a térképet árkategória (pl. csak 40 millió alatti vagy 80 millió feletti lakások), alapterület és szobaszám szerint.

**🔍 Mit érdemes kipróbálni?**
Állítsa be a szűrőt a „100 millió Ft feletti” kategóriára! Látni fogja, hogy a pontok szinte kivétel nélkül Óhegy legszebb zöldövezeti utcáira vagy az új építésű lakóparkokra szűkülnek le. Ezzel szemben a „35 millió alatti” szűrés szinte kizárólag az újhegyi garzonokat és a felújítandó kis téglalakásokat hagyja a térképen.

**💡 Fő tanulság**: A vizuális szűrés azonnal igazolja, hogy a kőbányai piac térben élesen szegregált.

In [5]:
w_szin = widgets.Dropdown(
    options=[('Négyzetméterár', 'nm_ar_huf'), ('Alapterület', 'alapterulet_nm'), ('Szobaszám', 'szobaszam_osszes'), ('Állapot Kód', 'allapot_kod')],
    value='nm_ar_huf',
    description='Színezés:'
)
w_meret = widgets.FloatSlider(min=4, max=15, step=1, value=7, description='Pontméret:')
w_tipus = widgets.RadioButtons(options=['Mindkettő', 'eladó', 'kiadó'], value='eladó', description='Típus:')

out_map = widgets.Output()

def frissit_terkep(*args):
    sub = df[df['minta_garantalt_pontos'] == 1].copy()
    if w_tipus.value != 'Mindkettő':
        mapped_val = 'elado' if w_tipus.value == 'eladó' else 'kiado'
        sub = sub[sub['listing_type'] == mapped_val]
        
    with out_map:
        clear_output(wait=True)
        fig = px.scatter_map(
            sub,
            lat='geokodolt_lat',
            lon='geokodolt_lon',
            color=w_szin.value,
            size_max=w_meret.value,
            hover_name='cim_teljes',
            zoom=12.2,
            center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
            map_style='carto-positron',
            title=f'Interaktív térkép: {w_szin.label} alapján (N={len(sub)})'
        )
        fig.update_layout(height=500, margin={"r":0,"t":40,"l":0,"b":0})
        fig.show()

w_szin.observe(frissit_terkep, names='value')
w_meret.observe(frissit_terkep, names='value')
w_tipus.observe(frissit_terkep, names='value')

display(widgets.HBox([w_szin, w_meret, w_tipus]))
display(out_map)
frissit_terkep()

Output()